# Deep Dive: Demystifying the Black Box

This notebook breaks open the pre-trained `ResNet18` model to physically look at its architecture, its hyperparameters, and its raw mathematical weights and biases!

<img src="resnet18_architecture.png" width="400"/>

ResNet18 has 18 Layers:
- Initial Convolution: 1 layer (conv1)
- Layer 1: 2 BasicBlocks × 2 Conv layers each (conv1 + conv2) = 4 layers
- Layer 2: 2 BasicBlocks × 2 Conv layers each (conv1 + conv2) = 4 layers
- Layer 3: 2 BasicBlocks × 2 Conv layers each (conv1 + conv2) = 4 layers
- Layer 4: 2 BasicBlocks × 2 Conv layers each (conv1 + conv2) = 4 layers
- Fully Connected: 1 layer (fc)

Total: 1 + 4 + 4 + 4 + 4 + 1 = 18 layers.

## 1. The "Big Picture" Architecture
The easiest way to see the hyperparameters (kernel size, stride, padding) is to print the model directly.

In [1]:
import os
## You need to set this environment variable for running the codes on Mac!!! Otherwise,
## your kernel may crash!!!
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torchvision.models as models

# 1. Hire the "College Graduate" (Load ResNet18)
resnet = models.resnet18(weights='DEFAULT')

# 2. Print the entire model architecture
print(resnet)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## 2. Extracting ALL Hyperparameters in Code
If you want to programmatically extract all settings (kernels, strides, padding, biases, etc.), you can loop through every layer.

In [2]:
import torch.nn as nn

print("--- ANALYZING RESNET18 ARCHITECTURE ---")

for name, layer in resnet.named_modules():
    
    if isinstance(layer, nn.Conv2d):
        print(f"[{name}] CONVOLUTION LAYER:")
        print(f"  - In Channels: {layer.in_channels}")
        print(f"  - Out Channels (Num Kernels): {layer.out_channels}")
        print(f"  - Kernel Size: {layer.kernel_size}")
        print(f"  - Stride: {layer.stride}")
        print(f"  - Padding: {layer.padding}")
        print(f"  - Has Bias?: {layer.bias is not None}\n")
        
    elif isinstance(layer, (nn.MaxPool2d, nn.AdaptiveAvgPool2d)):
        print(f"[{name}] POOLING LAYER:")
        if hasattr(layer, 'kernel_size'):
            print(f"  - Pool Size (Kernel): {layer.kernel_size}")
        if hasattr(layer, 'stride'):
            print(f"  - Stride: {layer.stride}")
        if hasattr(layer, 'padding'):
            print(f"  - Padding: {layer.padding}")
        print("\n")
        
    elif isinstance(layer, nn.Linear):
        print(f"[{name}] LINEAR CLASSIFIER HEAD:")
        print(f"  - Input Neurons: {layer.in_features}")
        print(f"  - Output Predictions: {layer.out_features}")
        print(f"  - Has Bias?: {layer.bias is not None}\n")


--- ANALYZING RESNET18 ARCHITECTURE ---
[conv1] CONVOLUTION LAYER:
  - In Channels: 3
  - Out Channels (Num Kernels): 64
  - Kernel Size: (7, 7)
  - Stride: (2, 2)
  - Padding: (3, 3)
  - Has Bias?: False

[maxpool] POOLING LAYER:
  - Pool Size (Kernel): 3
  - Stride: 2
  - Padding: 1


[layer1.0.conv1] CONVOLUTION LAYER:
  - In Channels: 64
  - Out Channels (Num Kernels): 64
  - Kernel Size: (3, 3)
  - Stride: (1, 1)
  - Padding: (1, 1)
  - Has Bias?: False

[layer1.0.conv2] CONVOLUTION LAYER:
  - In Channels: 64
  - Out Channels (Num Kernels): 64
  - Kernel Size: (3, 3)
  - Stride: (1, 1)
  - Padding: (1, 1)
  - Has Bias?: False

[layer1.1.conv1] CONVOLUTION LAYER:
  - In Channels: 64
  - Out Channels (Num Kernels): 64
  - Kernel Size: (3, 3)
  - Stride: (1, 1)
  - Padding: (1, 1)
  - Has Bias?: False

[layer1.1.conv2] CONVOLUTION LAYER:
  - In Channels: 64
  - Out Channels (Num Kernels): 64
  - Kernel Size: (3, 3)
  - Stride: (1, 1)
  - Padding: (1, 1)
  - Has Bias?: False

[layer2.

## 3. Looking at the Conv1 Weights (All 3 Channels)
Let's extract the raw decimal numbers for the very first 7x7 edge detector. Because input images have 3 color channels (RGB), this single edge detector is actually made of three 7x7 grids!

Change the codes (e.g. resnet.conv2) to view the raw numbers of other layers (e.g. resnet.conv2 to view info of Conv Layer 2)

In [3]:
print("--- LOOKING INSIDE THE RESNET18 FOR LEARNT WEIGHTS ---")

layer_of_interest = resnet.conv1
## Some sample codes here for your reference:
## layer_of_interest = resnet.layer2[0].conv2 # Second conv layer in first basic block of ResNet Layer 2.
## layer_of_interest = resnet.layer1[1].conv1 # First conv layer in second basic block of ResNet Layer 1.

weights = layer_of_interest.weight.data
print(f"Weights Shape: {weights.shape}")
# Shape is (64, 3, 7, 7)
# Meaning: 64 Kernels, looking at 3 RGB channels, using a 7x7 grid.

print(f"\n--- KERNEL 0: RED CHANNEL (7x7) ---")
print(weights[0, 0, :, :])

print(f"\n--- KERNEL 0: GREEN CHANNEL (7x7) ---")
print(weights[0, 1, :, :])

print(f"\n--- KERNEL 0: BLUE CHANNEL (7x7) ---")
print(weights[0, 2, :, :])


--- LOOKING INSIDE THE RESNET18 FOR LEARNT WEIGHTS ---
Weights Shape: torch.Size([64, 3, 7, 7])

--- KERNEL 0: RED CHANNEL (7x7) ---
tensor([[-0.0104, -0.0061, -0.0018,  0.0748,  0.0566,  0.0171, -0.0127],
        [ 0.0111,  0.0095, -0.1099, -0.2805, -0.2712, -0.1291,  0.0037],
        [-0.0069,  0.0591,  0.2955,  0.5872,  0.5197,  0.2563,  0.0636],
        [ 0.0305, -0.0670, -0.2984, -0.4387, -0.2709, -0.0006,  0.0576],
        [-0.0275,  0.0160,  0.0726, -0.0541, -0.3328, -0.4206, -0.2578],
        [ 0.0306,  0.0410,  0.0628,  0.2390,  0.4138,  0.3936,  0.1661],
        [-0.0137, -0.0037, -0.0241, -0.0659, -0.1507, -0.0822, -0.0058]])

--- KERNEL 0: GREEN CHANNEL (7x7) ---
tensor([[-1.1397e-02, -2.6619e-02, -3.4641e-02,  3.6812e-02,  3.2521e-02,
          6.6221e-04, -2.5743e-02],
        [ 4.5687e-02,  3.3603e-02, -1.0453e-01, -3.0885e-01, -3.1253e-01,
         -1.6051e-01, -1.2826e-03],
        [-8.3730e-04,  9.8420e-02,  4.0210e-01,  7.7035e-01,  7.0789e-01,
          3.6887e-01, 

## 4. The Fully Connected (FC) Layer
How many hidden layers are in ResNet18`s Fully Connected section? **Zero!**

In classic MLPs (like the one we built on Monday), we had multiple hidden FC layers. But in modern CNNs, 99% of the heavy lifting is done by the Convolutional Blocks! The FC section at the very end is literally just **one single linear layer** acting as the final voting head to classify the 1000 categories.

### Where do the 512 inputs come from?
Right before the data hits this FC layer, it goes through **Global Average Pooling**.
1. The final convolutional block (Layer 4) outputs 512 different "feature maps" (heatmaps).
2. The Global Average Pooling layer mathematically squashes each of those 512 heatmaps down into one single average number.
3. So, the massive 3D image data is completely flattened into a simple 1D array of exactly **512 numbers**.

Those 512 numbers are the direct inputs to this FC layer. That is why the weight matrix shape is `(1000, 512)`: **1000 output classes × 512 input numbers.**

Now that we know exactly where the inputs come from, let`s print out the actual voting weights in the code below!

In [4]:
print("--- FULLY CONNECTED (FC) LAYER ANALYSIS ---")
fc_layer = resnet.fc

# 1. Look at the FC WEIGHTS
fc_weights = fc_layer.weight.data
print(f"FC Weights Shape: {fc_weights.shape}") 
# Shape is (1000, 512)
# Meaning: 1000 output classes, each looking at 512 input features from the backbone.

print("\nHere are the first 10 weights connecting the 512 backbone features to Class 0:")
print(fc_weights[0, :10])

# 2. Look at the FC BIASES
fc_biases = fc_layer.bias.data
print(f"\nFC Biases Shape: {fc_biases.shape}")
# Shape is (1000)
# Meaning: 1 single bias number added to each of the 1000 output categories.

print("\nHere are the bias numbers for the 1000 output categories:")
print(fc_biases[:])


--- FULLY CONNECTED (FC) LAYER ANALYSIS ---
FC Weights Shape: torch.Size([1000, 512])

Here are the first 10 weights connecting the 512 backbone features to Class 0:
tensor([-0.0185, -0.0705, -0.0518, -0.0098,  0.0147, -0.0132, -0.0382,  0.2506,
        -0.0257, -0.0542])

FC Biases Shape: torch.Size([1000])

Here are the bias numbers for the 1000 output categories:
tensor([-2.6341e-03,  3.0005e-03,  6.5581e-04, -2.6909e-02,  6.3637e-03,
         1.3260e-02, -1.1178e-02,  2.0639e-02, -3.6373e-03, -1.2325e-02,
        -1.2629e-02, -7.2057e-03, -1.9321e-02, -2.4960e-02, -1.1885e-02,
        -8.3259e-03, -9.5745e-03, -1.6658e-02,  9.1804e-03, -1.5354e-02,
         7.1358e-03,  3.0737e-02,  1.3239e-02, -7.7528e-03,  4.7448e-03,
         1.1175e-02,  1.5949e-02, -1.6712e-02, -1.0130e-03, -3.7167e-03,
         6.5269e-03, -1.2041e-02,  9.0427e-03, -8.3279e-04,  8.8647e-03,
        -2.6307e-02, -1.4588e-02,  2.9433e-03,  2.9718e-03, -1.9125e-02,
        -4.7922e-03,  1.3828e-02,  9.8802e-03, 

## 5. Visualizing the Output: What does a $7 \times 7$ grid actually look like?
By the time the image reaches the end of Layer 4, it is no longer a physical photograph. It is a **$7 \times 7$ Heatmap**.

Because there are 512 channels, imagine a stack of 512 of these grids. If Channel #12 is a "Dog Nose Detector", its specific $7 \times 7$ tensor might look like this:
```python
tensor([[0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.12, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.00, 0.00, 0.85, 1.20, 0.00],
        [0.00, 0.00, 0.00, 0.00, 2.89, 3.14, 0.94], # <--- Huge numbers here!
        [0.00, 0.00, 0.00, 0.00, 1.10, 1.40, 0.00],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00]])
```
**How to read this:** The `0.00`s mean the feature is missing. But the massive `3.14` cluster means the network is screaming: *"I am highly confident that there is a dog nose right here, slightly below the center and off to the right side of the photo!"*

Meanwhile, Channel #13 (a "Car Wheel Detector") might be entirely `0.00` because there are no cars in the picture.

## 6. Tracking the Image Size (The Output Shapes)
Because the model dynamically processes whatever image size you give it, it doesn`t store "output shapes" in its weights. 

To see exactly how the image shrinks layer-by-layer (from $224 \times 224$ down to $7 \times 7$), we can use an awesome library called `torchinfo`. We pass a "fake" blank image of size `(1, 3, 224, 224)` into the network, and `torchinfo` tracks the exact mathematical dimensions at every single step!

In [5]:
# Install torchinfo (if you don`t have it already)
!pip install torchinfo -q

from torchinfo import summary

# We pass our resnet model, and a fake image shape: (Batch Size 1, 3 RGB Channels, 224 Width, 224 Height)
print("--- TRACKING IMAGE SHAPE THROUGH THE NETWORK ---")
summary(resnet, input_size=(1, 3, 224, 224), 
        col_names=["input_size", "output_size", "num_params"], 
        depth=3) # depth=3 lets us look inside the Residual Blocks


--- TRACKING IMAGE SHAPE THROUGH THE NETWORK ---


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
ResNet                                   [1, 3, 224, 224]          [1, 1000]                 --
├─Conv2d: 1-1                            [1, 3, 224, 224]          [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 112, 112]         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 64, 56, 56]           [1, 64, 56, 56]           --
│    └─BasicBlock: 2-1                   [1, 64, 56, 56]           [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3     

## 7. Cheat Sheet: Mapping `torchinfo` back to PyTorch Architecture
To help cross-reference the `torchinfo` indexing numbers back to the original PyTorch model layer names (which we printed in Cell 1), here is the exact translation of the main components:

- **`Conv2d: 1-1`** $\rightarrow$ `conv1` (Initial 7x7 Convolution)
- **`BatchNorm2d: 1-2`** $\rightarrow$ `bn1` (Initial Batch Normalization)
- **`ReLU: 1-3`** $\rightarrow$ `relu` (Initial Activation)
- **`MaxPool2d: 1-4`** $\rightarrow$ `maxpool` (Initial Pooling)
- **`Sequential: 1-5`** $\rightarrow$ `layer1` (The 1st Macro-Block of 4 Conv layers)
- **`Sequential: 1-6`** $\rightarrow$ `layer2` (The 2nd Macro-Block of 4 Conv layers)
- **`Sequential: 1-7`** $\rightarrow$ `layer3` (The 3rd Macro-Block of 4 Conv layers)
- **`Sequential: 1-8`** $\rightarrow$ `layer4` (The 4th Macro-Block of 4 Conv layers)
- **`AdaptiveAvgPool2d: 1-9`** $\rightarrow$ `avgpool` (The Global Average Pooling layer)
- **`Linear: 1-10`** $\rightarrow$ `fc` (The Final Fully Connected Classification Head)